# CPR AI Coach — Model Training Pipeline
**Project:** Capstone Project — African Leadership University, Kigali  
**Tagline:** *Buri mugenzi yagutabara* — Every bystander. Every second.

---
## What this notebook does
1. Mounts Google Drive and loads the CPR-Coach dataset (or generates dummy data if dataset not yet available)
2. Extracts 12-dim feature vectors from MediaPipe pose landmarks
3. Trains a BiLSTM classifier on 30-frame windows
4. Exports:
   - `cpr_classifier.tflite` → Flutter app (`assets/models/`)
   - `tfjs_model/model.json` + `.bin` shards → Web app (`web/assets/models/`)
5. Saves all exports to Google Drive

**Runtime:** GPU (T4 recommended). Training takes ~5–15 min on real data.

> ⚡ **No dataset yet?** Run all cells — the notebook will use synthetic dummy data so you can verify the full pipeline end-to-end.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q mediapipe==0.10.9 opencv-python-headless tensorflowjs==4.17.0 pyyaml
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

In [ ]:
# ── Cell 3: Imports and config ────────────────────────────────────────────────
import os, json, yaml, time, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Load config ───────────────────────────────────────────────────────────────
CONFIG_PATH = '/content/drive/MyDrive/capstone_data/config.yaml'
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    print('✓ Config loaded from Drive')
else:
    # Inline defaults (used when running fresh)
    cfg = {
        'dataset': {
            'landmarks_csv': '/content/drive/MyDrive/capstone_data/landmarks.csv',
            'export_dir':    '/content/drive/MyDrive/capstone_data/exports',
        },
        'classes': [
            'correct_compression','wrong_hand_high','wrong_hand_low',
            'bent_elbows','too_shallow','rate_too_slow','rate_too_fast','not_compressing'
        ],
        'features': {'n_features': 12, 'window_size': 30},
        'model': {
            'lstm_units': [128, 64], 'dropout': 0.40,
            'batch_size': 32, 'epochs': 60, 'lr': 0.001,
            'val_split': 0.15, 'test_split': 0.10,
            'early_stop_patience': 10
        },
        'export': {
            'tflite_filename': 'cpr_classifier.tflite',
            'tfjs_dir': 'tfjs_model',
            'quantize_int8': True
        }
    }
    print('ℹ Using inline defaults (config.yaml not found in Drive)')

CLASS_LABELS  = cfg['classes']
N_CLASSES     = len(CLASS_LABELS)
N_FEATURES    = cfg['features']['n_features']   # 12
WINDOW        = cfg['features']['window_size']  # 30
EXPORT_DIR    = Path(cfg['dataset']['export_dir'])
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Classes: {N_CLASSES}  |  Features: {N_FEATURES}  |  Window: {WINDOW}')
print(f'Export dir: {EXPORT_DIR}')
print(f'GPU: {tf.test.gpu_device_name() or "none — training on CPU"}')

In [ ]:
# ── Cell 4: Feature extraction helpers (mirrors landmarkMath.dart / .js) ─────

def angle_degrees(a, b, c):
    """Angle at joint B (vectors B→A and B→C), degrees."""
    ba = a - b;  bc = c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

# MediaPipe landmark indices
LM = dict(
    NOSE=0, L_SHLDR=11, R_SHLDR=12,
    L_ELBOW=13, R_ELBOW=14,
    L_WRIST=15, R_WRIST=16,
    L_HIP=23,  R_HIP=24,
)

def norm(v, lo, hi):
    return np.clip((v - lo) / (hi - lo + 1e-8), 0, 1)

def extract_features(row, prev_wrist_y=0.5, prev_vel_y=0.0,
                     baseline_wrist_y=0.5, ref_shoulder_w=0.3):
    """
    Extract 12-dim feature vector from a MediaPipe landmark row.
    `row` must contain columns like x_11, y_11, z_11, vis_11 ... for each landmark.
    Returns (feature_vector, new_wrist_y, new_vel_y).
    """
    def lm(idx):
        return np.array([row.get(f'x_{idx}', 0), row.get(f'y_{idx}', 0)])

    ls = lm(LM['L_SHLDR']); rs = lm(LM['R_SHLDR'])
    le = lm(LM['L_ELBOW']); re = lm(LM['R_ELBOW'])
    lw = lm(LM['L_WRIST']); rw = lm(LM['R_WRIST'])
    lh = lm(LM['L_HIP']);   rh = lm(LM['R_HIP'])

    elb_l   = angle_degrees(ls, le, lw)
    elb_r   = angle_degrees(rs, re, rw)
    elb_m   = (elb_l + elb_r) / 2

    mid_shldr = (ls + rs) / 2
    mid_hip   = (lh + rh) / 2
    diff      = mid_shldr - mid_hip
    spine     = np.degrees(np.arctan2(abs(diff[0]), abs(diff[1]) + 1e-8))

    wrist_y   = (lw[1] + rw[1]) / 2
    sw        = np.linalg.norm(ls - rs)
    vel_y     = wrist_y - prev_wrist_y
    acc_y     = vel_y - prev_vel_y
    depth     = np.clip(abs(wrist_y - baseline_wrist_y) / (ref_shoulder_w + 1e-8), 0, 1)

    vis = [row.get(f'vis_{i}', 1.0) for i in [LM['L_SHLDR'],LM['R_SHLDR'],
                                                LM['L_ELBOW'],LM['R_ELBOW'],
                                                LM['L_WRIST'],LM['R_WRIST']]]
    mean_vis = np.mean(vis)

    fv = [
        norm(elb_l, 0, 180),
        norm(elb_r, 0, 180),
        norm(elb_m, 0, 180),
        norm(spine,  0,  90),
        wrist_y,
        np.clip(vel_y, -1, 1),
        np.clip(acc_y, -1, 1),
        depth,
        norm(sw, 0.05, 0.6),
        mean_vis,
        float(row.get(f'vis_{LM["L_WRIST"]}', 1.0) >= 0.6),
        float(row.get(f'vis_{LM["R_WRIST"]}', 1.0) >= 0.6),
    ]
    return np.array(fv, dtype=np.float32), wrist_y, vel_y

print('✓ Feature extraction helpers ready')

In [ ]:
# ── Cell 5: Load dataset (or generate synthetic dummy data) ───────────────────

LANDMARKS_CSV = cfg['dataset']['landmarks_csv']
USE_DUMMY     = not os.path.exists(LANDMARKS_CSV)

if USE_DUMMY:
    print('⚠  landmarks.csv not found — generating SYNTHETIC dummy data')
    print('   This is for pipeline testing only.')
    print('   Replace with real CPR-Coach landmark data for production training.\n')

    SAMPLES_PER_CLASS = 200
    rng = np.random.default_rng(42)
    rows = []

    for class_idx, label in enumerate(CLASS_LABELS):
        for _ in range(SAMPLES_PER_CLASS):
            row = {'label': label}
            for lm_idx in range(33):
                row[f'x_{lm_idx}']   = rng.uniform(0.1, 0.9)
                row[f'y_{lm_idx}']   = rng.uniform(0.1, 0.9)
                row[f'z_{lm_idx}']   = rng.uniform(-0.5, 0.5)
                row[f'vis_{lm_idx}'] = rng.uniform(0.7, 1.0)

            # Add class-specific signal so the model can learn something
            if label == 'correct_compression':
                row['x_13'] = 0.35 + rng.normal(0, 0.02)  # L elbow inside
                row['x_14'] = 0.65 + rng.normal(0, 0.02)  # R elbow inside
            elif label == 'bent_elbows':
                row['y_13'] = row['y_11'] - 0.12 + rng.normal(0, 0.02)  # bent
                row['y_14'] = row['y_12'] - 0.12 + rng.normal(0, 0.02)
            elif label == 'wrong_hand_high':
                row['y_15'] = row['y_11'] - 0.15 + rng.normal(0, 0.02)  # hands high
                row['y_16'] = row['y_12'] - 0.15 + rng.normal(0, 0.02)
            rows.append(row)

    df = pd.DataFrame(rows)
    print(f'  Generated {len(df)} frames across {N_CLASSES} classes')
else:
    df = pd.read_csv(LANDMARKS_CSV)
    print(f'✓ Loaded {len(df)} frames from {LANDMARKS_CSV}')

print(df['label'].value_counts().to_string())

In [ ]:
# ── Cell 6: Build windowed sequences ─────────────────────────────────────────

def build_sequences(df, window_size, class_labels):
    """Slide a window over per-class frame sequences to produce (X, y) arrays."""
    X_seqs, y_seqs = [], []
    label_enc = {l: i for i, l in enumerate(class_labels)}

    for label in class_labels:
        frames = df[df['label'] == label].reset_index(drop=True)
        if len(frames) < window_size:
            print(f'  ⚠  {label}: only {len(frames)} frames — skipping')
            continue

        # Extract features frame-by-frame with running state
        prev_wy, prev_vy = 0.5, 0.0
        baseline_wy      = float(frames.iloc[0].get('y_15', 0.5))
        ref_sw           = 0.3
        feature_rows     = []

        for _, row in frames.iterrows():
            fv, prev_wy, prev_vy = extract_features(
                row, prev_wy, prev_vy, baseline_wy, ref_sw
            )
            feature_rows.append(fv)

        feature_rows = np.array(feature_rows)  # (n_frames, 12)

        for i in range(0, len(feature_rows) - window_size + 1, window_size // 2):
            X_seqs.append(feature_rows[i : i + window_size])
            y_seqs.append(label_enc[label])

    return np.array(X_seqs, dtype=np.float32), np.array(y_seqs, dtype=np.int32)

print('Building windowed sequences...')
X, y = build_sequences(df, WINDOW, CLASS_LABELS)
print(f'  X shape: {X.shape}  |  y shape: {y.shape}')
print(f'  Sequences per class: {np.bincount(y)}')

In [ ]:
# ── Cell 7: Train / val / test split ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

test_size = cfg['model']['test_split']
val_size  = cfg['model']['val_split'] / (1 - test_size)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_size, stratify=y_trainval, random_state=42
)

print(f'Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}')

In [ ]:
# ── Cell 8: Build BiLSTM model ────────────────────────────────────────────────

def build_model(window, n_features, n_classes, lstm_units, dropout, lr):
    inp = tf.keras.Input(shape=(window, n_features), name='input')

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(lstm_units[0], return_sequences=True, name='bilstm_1')
    )(inp)
    x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(lstm_units[1], return_sequences=False, name='bilstm_2')
    )(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    x   = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = tf.keras.Model(inputs=inp, outputs=out, name='cpr_classifier')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_model(
    window=WINDOW, n_features=N_FEATURES, n_classes=N_CLASSES,
    lstm_units=cfg['model']['lstm_units'],
    dropout=cfg['model']['dropout'],
    lr=cfg['model']['lr'],
)
model.summary()

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────────

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=cfg['model']['early_stop_patience'],
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5
    ),
]

start = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=cfg['model']['epochs'],
    batch_size=cfg['model']['batch_size'],
    callbacks=callbacks,
    verbose=1,
)
print(f'\nTraining complete in {(time.time()-start)/60:.1f} min')

In [ ]:
# ── Cell 10: Training curves ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('CPR Classifier — Training History', fontsize=14)

axes[0].plot(history.history['accuracy'],     label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_ylim(0, 1)

axes[1].plot(history.history['loss'],     label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout()
curves_path = EXPORT_DIR / 'training_curves.png'
plt.savefig(curves_path, dpi=100)
plt.show()
print(f'✓ Saved training curves → {curves_path}')

In [ ]:
# ── Cell 11: Evaluate on test set ─────────────────────────────────────────────

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc*100:.1f}%   |   Test loss: {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print('\n── Classification Report ──')
print(classification_report(y_test, y_pred, target_names=CLASS_LABELS, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
cm_path = EXPORT_DIR / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=100)
plt.show()
print(f'✓ Saved confusion matrix → {cm_path}')

In [ ]:
# ── Cell 12: Export TFLite (INT8 quantised for mobile) ───────────────────────

def export_tflite(model, export_dir, filename, quantize_int8, X_rep):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantize_int8:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        def representative_dataset():
            for i in range(min(200, len(X_rep))):
                yield [X_rep[i:i+1]]

        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type  = tf.float32  # Keep float I/O for app compatibility
        converter.inference_output_type = tf.float32

    tflite_model = converter.convert()
    out_path = Path(export_dir) / filename
    with open(out_path, 'wb') as f:
        f.write(tflite_model)

    size_kb = len(tflite_model) / 1024
    print(f'✓ TFLite saved → {out_path}  ({size_kb:.1f} KB)')
    return out_path

tflite_path = export_tflite(
    model, EXPORT_DIR,
    cfg['export']['tflite_filename'],
    cfg['export']['quantize_int8'],
    X_train[:200],
)

In [ ]:
# ── Cell 13: Verify TFLite model ──────────────────────────────────────────────

interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
in_det  = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

print(f'Input  shape: {in_det["shape"]}   dtype: {in_det["dtype"]}')
print(f'Output shape: {out_det["shape"]}  dtype: {out_det["dtype"]}')

# Spot-check one test example
dummy = X_test[:1].copy()
interp.set_tensor(in_det['index'], dummy)
interp.invoke()
probs = interp.get_tensor(out_det['index'])[0]
pred_label = CLASS_LABELS[np.argmax(probs)]
true_label = CLASS_LABELS[y_test[0]]
print(f'\nSpot check — True: {true_label}  |  Predicted: {pred_label}  |  Confidence: {max(probs)*100:.1f}%')
print('✓ TFLite model verified')

In [ ]:
# ── Cell 14: Export TensorFlow.js model (for web app) ────────────────────────

import tensorflowjs as tfjs

tfjs_dir = EXPORT_DIR / cfg['export']['tfjs_dir']
tfjs_dir.mkdir(parents=True, exist_ok=True)

# Save Keras .h5 first, then convert to TFJS graph model
h5_path = EXPORT_DIR / 'cpr_classifier.h5'
model.save(h5_path)

tfjs.converters.save_keras_model(model, str(tfjs_dir))
print(f'✓ TFJS model saved → {tfjs_dir}')

# Show exported files
for f in sorted(tfjs_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:.1f} KB')

In [ ]:
# ── Cell 15: Save model metadata JSON ────────────────────────────────────────

metadata = {
    'model_name':   'cpr_classifier',
    'version':      '1.0.0',
    'trained_at':   time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'dataset':      'CPR-Coach sample' if not USE_DUMMY else 'SYNTHETIC (dummy)',
    'n_frames':     len(X),
    'test_accuracy': float(f'{test_acc:.4f}'),
    'input_shape':  [1, WINDOW, N_FEATURES],
    'output_shape': [1, N_CLASSES],
    'class_labels': CLASS_LABELS,
    'feature_names': [
        'left_elbow_angle_norm', 'right_elbow_angle_norm', 'mean_elbow_angle_norm',
        'spine_verticality_norm', 'mean_wrist_y', 'wrist_y_velocity',
        'wrist_y_acceleration', 'compression_depth', 'shoulder_width_norm',
        'mean_visibility', 'left_wrist_visible', 'right_wrist_visible',
    ],
    'normalisation': {
        'elbow_angle': {'min': 0, 'max': 180},
        'spine':       {'min': 0, 'max': 90},
        'shoulder_w':  {'min': 0.05, 'max': 0.60},
    },
    'thresholds': {
        'confidence': 0.70,
        'bpm_min':    100,
        'bpm_max':    120,
    }
}

meta_path = EXPORT_DIR / 'cpr_classifier_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✓ Metadata saved → {meta_path}')
print(json.dumps(metadata, indent=2))

In [ ]:
# ── Cell 16: Final summary ────────────────────────────────────────────────────

print('=' * 60)
print('  CPR AI Coach — Training Complete')
print('=' * 60)
print(f'  Dataset:         {"SYNTHETIC" if USE_DUMMY else "CPR-Coach"}')
print(f'  Sequences:       {len(X)}')
print(f'  Test accuracy:   {test_acc*100:.1f}%')
print()
print('  Export files:')
print(f'    TFLite:  {tflite_path}')
print(f'    TFJS:    {tfjs_dir}/model.json')
print(f'    Keras:   {h5_path}')
print(f'    Meta:    {meta_path}')
print()
print('  Next steps:')
print('  1. Download cpr_classifier.tflite')
print('     → place in  capstone_project/assets/models/')
print('  2. Download tfjs_model/ folder')
print('     → place in  capstone_project/web/assets/models/')
print('  3. Run  flutter pub get  then  flutter run')
print('  4. Run  cd web && npm run dev')
print('=' * 60)

if USE_DUMMY:
    print()
    print('  ⚠  IMPORTANT: This model was trained on SYNTHETIC data.')
    print('     Accuracy will be low on real CPR footage.')
    print('     Collect real CPR-Coach data and retrain for deployment.')